# Get Summary + Analysis Tables

## Set up

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../"))

import pandas as pd
from src.config import BASE_PATH, SEED
from src.data_utils import get_feature_lists, export_data
from src.summary_analysis import get_summary_analysis_table, impute_numerical

In [ ]:
stats_df_path = BASE_PATH / "data" / "processed" / "df_used_for_sum_analysis.parquet"
base_df_path = BASE_PATH / "data/raw/cleaned/NSQIP_mast_combined_cancer.parquet"
table_export_path = BASE_PATH / "results/summary_analysis.xlsx"

## Prep df

- Note that some vars in this section are needed for the next

In [ ]:
reorder_cols_list = [
    ############ TABLE 1 ############
    ## Demographics
    "AGE",
    "BMI",
    "SEX",
    "RACE",
    "HISPANIC",
    ## Pre-op health + comorbidities
    "DIABETES",
    "SMOKE",
    "DYSPNEA",  # missing 21-24
    "VENTILAT",
    "HXCOPD",
    "ASCITES",
    "HXCHF",
    "HYPERMED",
    "RENAFAIL",  # missing 21
    "DIALYSIS",
    "DISCANCR",
    "WNDINF",  # missing 21-24
    "STEROID",
    "WTLOSS",  # missing 21-24
    "BLEEDDIS",
    "TRANSFUS",
    "PRSEPIS",  # not included in modeling
    "FNSTATUS2",  # not included in modeling
    ## BLOOD
    "PRALBUM",
    "PRWBC",
    "PRHCT",
    "PRPLATE",
    "ASACLAS",
    ## INTRA-OP
    "SURGINDICD",
    "OPTIME",  # not used for modeling
    "INOUT",
    "URGENCY",
    "SURGSPEC",
    "ANESTHES",
    "OPERYR",
    #### CPT OP ####
    ## Resection
    "PARTIALCPT",
    "SUBSIMPLECPT",
    "RADICALCPT",
    "MODIFIEDRADICALCPT",
    ## axillary
    "SNLBCPT",
    "ALNDCPT",
    "NOLYMPH",  # not included in modeling, just to make clean patients w/o any
    ## implant-based
    "IMMEDIATECPT",
    "DELAYEDCPT",
    "TEINSERTIONCPT",
    "TEEXPANDERCPT",
    ## autologous
    "FREECPT",
    "LATCPT",
    "SINTRAMCPT",
    "SINTRAMSUPERCPT",
    "BITRAMCPT",
    ## adjunct + revision
    "AUGPROSIMPCPT",
    "MASTOCPT",
    "BREASTREDCPT",
    "FATGRAFTCPT",
    "REVRECBREASTCPT",
    "ADJTISTRANSCPT",
    "NPWTCPT",
    "OTHERRECONTECHCPT",
    ## Additional
    "PROCANATCPT",
    "SURGTIMINGCPT",
    ############ TABLE 3 ############
    "TOTHLOS",
    "DISCHDEST",
    "ANY",
    "SERIOUS",
    "MORT",
    "UNPLNREOP",
    "READ",
    "UNPLNREAD",
    ### Surgical Complications
    "SSI",
    "SUPINFEC",
    "WNDINFD",
    "ORGSPCSSI",
    "DEHIS",
    ## other
    "OTHBLEED",
    ## VTE Complications
    "VTE",
    "OTHDVT",
    "PULEMBOL",
    ## CARDIAC
    "CARDIAC_COMP",
    "CDARREST",
    "CDMI",
    ## Sepsis
    "SEPSIS",
    "OTHSYSEP",
    "OTHSESHOCK",
    ## RENAL
    "RENAL",
    "RENAINSF",  # missing 21
    "OPRENAFL",
    ## Respiratory
    "FAILWEAN",
    "REINTUB",
    "PNEUMO",
    ## other Medical Complications
    "UTI",
    "CNSCVA",
]

In [ ]:
df = pd.read_parquet(base_df_path)
df["BMI"] = (df["WEIGHT"] / (df["HEIGHT"] ** 2)) * 703
df = df.drop(["WEIGHT", "HEIGHT"], axis=1)

Subset 2014-2024

In [ ]:
df = df[df["OPERYR"].isin(range(2014, 2025))]
# Dropping 2008-2010 removes 'Unknown_08_10', leaving READ/UNPLNREAD as pure
# Yes/No -> encode 0/1 so they're treated like the other binary cols
df[["READ", "UNPLNREAD"]] = (df[["READ", "UNPLNREAD"]] == "Yes").astype(int)

In [ ]:
ordered_df = df[reorder_cols_list].copy()

In [ ]:
## Get feature lists
feature_dict = get_feature_lists(ordered_df)
# consider operation yr categorical for analysis
feature_dict["numerical_cols"].remove("OPERYR")
feature_dict["ordinal_cols"].append("OPERYR")
num_cols = feature_dict["numerical_cols"]
impute_cols = num_cols + ["ASACLAS"]  # add ASA class to impute

In [ ]:
# num (%) missing for numeric
num_missing_dict = {}
tot = len(ordered_df)
for col in num_cols:
    n_missing = int(ordered_df[col].isna().sum())
    perc_missing = round(100 * n_missing / tot, 1)
    if n_missing > 0 and perc_missing < 0.1:
        perc_missing = "<0.1"
    num_missing_dict[col] = f"{n_missing} ({perc_missing})"
num_missing_dict

## Impute

- Note this section only needs to be run once

In [ ]:
df_impute = impute_numerical(df=ordered_df, impute_cols=impute_cols, seed=SEED)
## Round ASACLASS imputations back to whole numbers
df_impute["ASACLAS"] = df_impute["ASACLAS"].round(0).astype(float)
assert df_impute.isna().sum().sum() == 0

Export df used for stats

In [ ]:
export_data(df_impute, stats_df_path)

## Run summary + analysis

In [ ]:
df_impute = pd.read_parquet(stats_df_path)
outcome_sub_cols = {
    "SERIOUS": [
        "CDARREST",  # cardiac arrest
        "CDMI",  # myocaridal
        "OUPNEUMO",  # pneumonia
        "RENAINSF",  # missing 21 --> progressive renal insufficiency
        "OPRENAFL",  # acute renal failure
        "PULEMBOL",  # PE
        "OTHDVT",  # venous thrombosis
        "UNPLNREOP",  # unplanned reop
        "WNDINFD",  # deep incisional SSI
        "ORGSPCSSI",  # organ space SSI
        "OTHSYSEP",  # post-op sepsis (systemic sepsis is preo-op)
        "REINTUB",  # unplanned intubation
        "URNINFEC",  # UTI
        "DEHIS",  # wound disruption
    ],
    "ANY": [
        "CDARREST",  # cardiac arrest
        "CDMI",  # myocaridal
        "OUPNEUMO",  # pneumonia
        "RENAINSF",  # missing 21 --> progressive renal insufficiency
        "OPRENAFL",  # acute renal failure
        "PULEMBOL",  # PE
        "OTHDVT",  # venous thrombosis
        "UNPLNREOP",  # unplanned reop
        "WNDINFD",  # deep incisional SSI
        "ORGSPCSSI",  # organ space SSI
        "OTHSYSEP",  # post-op sepsis (systemic sepsis is preo-op)
        "REINTUB",  # unplanned intubation
        "URNINFEC",  # UTI
        "DEHIS",  # wound disruption
        "SUPINFEC",  # superficial SSI
        "FAILWEAN",  # post-op vent > 48H
        "CNSCVA",  # stroke
    ],
    "PNEUMO": ["PNEUMO"],
    "CARDIAC_COMP": [
        "CDARREST",  # cardiac arrest
        "CDMI",  # myocaridal
    ],
    "VTE": ["OTHDVT", "PULEMBOL"],
    "SEPSIS": [
        "OTHSYSEP",  # post-op sepsis
        "OTHSESHOCK",  # post-op septic shock
    ],
    "SSI": [
        "SUPINFEC",  # superficial SSI
        "WNDINFD",  # deep incisional SSI
        "ORGSPCSSI",  # organ space SSI
    ],
    "UTI": ["UTI"],
    "RENAL": [
        "OPRENAFL",  # acute renal failure
        "RENAINSF",  # missing 21 --> progressive renal insufficiency
    ],
    "UNPLNREOP": ["UNPLNREOP"],
    "MORT": ["MORT"],
}

# map binary encodings back to their labels if not yes/no
bin_cat_dict = {
    "SEX": {
        1: "Female",
        0: "Male",
    },
    "HISPANIC": {1: "Hispanic", 0: "Not Hispanic/Unknown"},
    "INOUT": {
        1: "Inpatient",
        0: "Outpatient",
    },
    "URGENCY": {1: "Urgent/Emergent", 0: "Elective"},
    "SURGINDICD": {1: "Malignangt", 0: "Carcinoma"},
}

In [ ]:
# all possible entries
all_categories = {}
for col in df_impute.columns:
    if col not in num_cols:
        ## Dict: {column_names: [<unique_entries>]}
        if col in bin_cat_dict.keys():
            all_categories[col] = list(bin_cat_dict[col].values())
        else:
            all_categories[col] = list(df_impute[col].unique())

In [ ]:
summary_analysis_table = get_summary_analysis_table(
    outcome_list=list(outcome_sub_cols.keys()),
    df=df_impute,
    all_categories=all_categories,
    bin_cat_dict=bin_cat_dict,
    num_missing_dict=num_missing_dict,
    outcome_sub_cols=outcome_sub_cols,
    feature_dict=feature_dict,
    verbose=True,
)
summary_analysis_table

In [ ]:
export_data(data_to_export=summary_analysis_table, export_path=table_export_path)